# Beginner-Friendly RAG Tutorial

Welcome to this simple tutorial on **Retrieval-Augmented Generation (RAG)**! 

RAG is a technique that gives Language Models (like ChatGPT) access to your own private data. Instead of training the model on your data (which is expensive and difficult), you:
1. Break your documents into chunks.
2. Search for the most relevant chunks based on a user's question.
3. Pass those relevant chunks to the LLM to generate an answer.

### 1. Install and Import Dependencies
Let's install the libraries we need:
- `langchain`: A framework for building LLM applications.
- `sentence-transformers`: Open-source models to create embeddings.
- `faiss-cpu`: A super fast vector database by Facebook to store and retrieve chunks.

In [ ]:
!pip install -q langchain langchain-community sentence-transformers faiss-cpu

import warnings
warnings.filterwarnings('ignore')

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.docstore.document import Document

### 2. Load and Split the Corpus
LLMs can only process a certain amount of text at a time. If we have a massive book or PDF, we must break it down. We'll use LangChain's `RecursiveCharacterTextSplitter` to chop our text into paragraphs.

In [ ]:
# Sample Knowledge Base (Our Corpus)
sample_text = """
The Kingdom of Asgardia is a fictional kingdom where dragons roam the skies and magic is taught in schools.
Its capital city, Valerion, is famous for its towering crystal spires. 
In the year 2045, the Great Dragon War began, led by the wicked sorcerer Malakor.
Malakor was eventually defeated by the hero Elyndor, who wielded the legendary Sunblade.
Elyndor established a peaceful era that lasted for three centuries.
"""

# Convert string to a LangChain Document
document = Document(page_content=sample_text)

# We split the text so we don't pass the whole thing to the LLM. 
# Here, chunks of ~100 characters seem reasonable given our short text.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_documents([document])

print(f"Split document into {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk.page_content}")

### 3. Create Embeddings and Vector Store
Instead of doing a keyword text search (like "Ctrl+F"), RAG searches for meaning.
We do this by turning text into arrays of numbers called **Embeddings** using a machine learning model.
Similar sentences will have similar arrays of numbers!

We will use an open-source huggingface model: `all-MiniLM-L6-v2`. We store these embeddings in **FAISS**, a vector database.

In [ ]:
# 1. Download an open source embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Add all of our text chunks to the FAISS Vector Database
vector_store = FAISS.from_documents(chunks, embeddings)

# Test a search without the LLM answering yet
query = "Who defeated Malakor?"
relevant_docs = vector_store.similarity_search(query, k=2)

print("Our Vector DB found these chunks matching your query:")
for d in relevant_docs:
    print(f"- {d.page_content}")

### 4. Initialize the Language Model
Now that we can search our database for the paragraphs that answer the user's question, we need an **LLM** (Large Language Model) to take those paragraphs and generate a human-readable summary.

To keep it free and without API keys, we'll download a small open-source instruction model (`google/flan-t5-small`). You can easily swap this out with OpenAI's ChatGPT models or Google's Gemini Models if you have an API key!

In [ ]:
!pip install -q transformers 
from transformers import pipeline
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

# Download a tiny model that will run fast locally on your computer/colab
generate_text = pipeline(model="google/flan-t5-small", task="text2text-generation", max_length=100)

# Connect it to LangChain
llm = HuggingFacePipeline(pipeline=generate_text)

# Test the LLM directly (without RAG context)
print(llm.invoke("What is the capital of France?"))

### 5. Build and Test the RAG Chain
We bring the two components together:
1. **The Retriever** queries the `FAISS` Vector DB.
2. **The Output** of the Retriever is fed straight to the **LLM** so it can give us an answer.

In LangChain, this built-in connection is called an `RetrievalQA` or `create_retrieval_chain`. We'll use the modern `create_retrieval_chain` approach.

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Turn our vector store into a tool that fetches top chunks
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# 2. Tell the LLM how to answer
system_prompt = (
    "You are an assistant for answering questions about the fictional kingdom of Asgardia. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know."
    "\n\nContext:"
    "{context}"
)

# 3. Create the Instruction Template
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 4. Chain it all together
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 5. TEST IT!
my_question = "Who is Malakor and what happened to him?"
response = rag_chain.invoke({"input": my_question})

print("Question:", my_question)
print("Answer:", response["answer"])